# Studying the Final Neutron Star Population

In [ ]:
import astropy.coordinates as coord
import astropy.units as u
import numpy as np
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy.optimize import curve_fit
from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import mlpoppyns.simulator.multiband_emission.emission_radio as er

import utilities.plot_settings

## Loading simulated data

Select a `final_population.pkl.gz` file to import:

In [ ]:
path_to_simulation = pathlib.Path("../../data/example_simulation_full_edm/")

data_full = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "final_population.pkl.gz"),
    compression="gzip",
)


data_PMPS = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)


data_SMPS = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)

data_HTRU_low_mid = pd.read_pickle(
    pathlib.Path().joinpath(
        path_to_simulation, "survey_HTRU_low_mid_results.pkl.gz"
    ),
    compression="gzip",
)

data_HTRU_high = pd.read_pickle(
    pathlib.Path().joinpath(
        path_to_simulation, "survey_HTRU_high_results.pkl.gz"
    ),
    compression="gzip",
)

data_full.columns

In [ ]:
r = data_full["r"]["[kpc]"].to_numpy()
phi = data_full["phi"]["[rad]"].to_numpy()
z = data_full["z"]["[kpc]"].to_numpy()
RA = data_full["ra"]["[deg]"].to_numpy()
DEC = data_full["dec"]["[deg]"].to_numpy()
pm_RA = data_full["pm_ra"]["[mas yr^-1]"].to_numpy()
pm_DEC = data_full["pm_dec"]["[mas yr^-1]"].to_numpy()
v_r = data_full["v_r"]["[km s^-1]"].to_numpy()
v_phi = data_full["v_phi"]["[km s^-1]"].to_numpy()
v_z = data_full["v_z"]["[km s^-1]"].to_numpy()
dist = data_full["dist"]["[kpc]"].to_numpy()
B = data_full["B"]["[G]"].to_numpy()
chi = data_full["chi"]["[rad]"].to_numpy()
P = data_full["P"]["[s]"].to_numpy()
P_dot = data_full["P_dot"]["[s s^-1]"].to_numpy()
L_radio_bol = data_full["L_radio_bol"]["[erg s^-1]"].to_numpy()
w_int = data_full["w_int"]["[s]"].to_numpy()
intercepted_radio = data_full["intercepted_radio"][" "].to_numpy(dtype=bool)
age = data_full["age"]["[yr]"].to_numpy()
spectral_index = data_full["spectral_index"].to_numpy()
x = r * np.cos(phi)
y = r * np.sin(phi)

In [ ]:
idx_PMPS = data_PMPS["idx"].to_numpy(dtype=int)
S_radio_PMPS = data_PMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_PMPS = data_PMPS["w_eff"]["[s]"].to_numpy()

idx_SMPS = data_SMPS["idx"].to_numpy(dtype=int)
S_radio_SMPS = data_SMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_SMPS = data_SMPS["w_eff"]["[s]"].to_numpy()

idx_HTRU_low_mid = data_HTRU_low_mid["idx"].to_numpy(dtype=int)
S_radio_HTRU_low_mid = data_HTRU_low_mid["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_HTRU_low_mid = data_HTRU_low_mid["w_eff"]["[s]"].to_numpy()

idx_HTRU_high = data_HTRU_high["idx"].to_numpy(dtype=int)
S_radio_HTRU_high = data_HTRU_high["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_HTRU_high = data_HTRU_high["w_eff"]["[s]"].to_numpy()

set_PMPS = set(idx_PMPS.tolist())
set_SMPS = set(idx_SMPS.tolist())
set_HTRU_low_mid = idx_HTRU_low_mid.tolist()
set_HTRU_high = idx_HTRU_high.tolist()

idx_SMPS_noduplicates = list(set_SMPS - set_PMPS)

idx_radio_det = idx_PMPS.tolist() + idx_SMPS_noduplicates

A few statistics on the detected number of stars.

In [ ]:
number_intercepted = len(intercepted_radio[intercepted_radio == True])
number_detected_PMPS = len(idx_PMPS)
number_detected_SMPS = len(idx_SMPS)
number_detected_HTRU_low_mid = len(idx_HTRU_low_mid)
number_detected_HTRU_high = len(idx_HTRU_high)
print(
    f"Total number of detected pulsars in radio: {number_detected_PMPS + number_detected_SMPS + number_detected_HTRU_low_mid + number_detected_HTRU_high}"
)

fraction_intercepted = len(intercepted_radio[intercepted_radio == True]) / len(
    intercepted_radio
)
fraction_detected_PMPS = len(idx_PMPS) / len(intercepted_radio)
fraction_detected_SMPS = len(idx_SMPS) / len(intercepted_radio)
fraction_detected_HTRU_low_mid = len(idx_HTRU_low_mid) / len(intercepted_radio)
fraction_detected_HTRU_high = len(idx_HTRU_high) / len(intercepted_radio)

print(
    f"Fraction of pulsars pointing at us: {fraction_intercepted}, ({number_intercepted}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by PMPS: {fraction_detected_PMPS}, ({number_detected_PMPS}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by SMPS: {fraction_detected_SMPS}, ({number_detected_SMPS}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by HTRU low and mid latitude: {fraction_detected_HTRU_low_mid}, ({number_detected_HTRU_low_mid}/{len(intercepted_radio)})"
)
print(
    f"Fraction of pulsars detected by HTRU high latitude: {fraction_detected_HTRU_high}, ({number_detected_HTRU_high}/{len(intercepted_radio)})"
)

## Age information

In [ ]:
age_bins = np.linspace(cfg["t_age_min"], cfg["t_age_max"], 31)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    age,
    bins=age_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Simulated all",
)
ax.hist(
    age[intercepted_radio],
    bins=age_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    alpha=1,
    label="Intercepting our LOS",
)
ax.hist(
    age[idx_PMPS],
    bins=age_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="Detected by PMPS",
)
ax.hist(
    age[idx_SMPS],
    bins=age_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Detected by SMPS",
)
ax.hist(
    age[idx_HTRU_low_mid],
    bins=age_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label="Detected by HTRU low and mid",
)
ax.hist(
    age[idx_HTRU_high],
    bins=age_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    alpha=1,
    label="Detected by HTRU high",
)

plt.xlabel(r"Age [yr]")
plt.ylabel(r"Number of NSs")
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

## Positional information

Top view of the Galactic plane.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulated all",
)

ax.plot(
    x[intercepted_radio],
    y[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    x[idx_PMPS],
    y[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    x[idx_SMPS],
    y[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    x[idx_HTRU_low_mid],
    y[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.plot(
    x[idx_HTRU_high],
    y[idx_HTRU_high],
    linestyle="None",
    marker="o",
    color="tab:olive",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU high",
)

ax.plot(
    0.0,
    8.3,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=6,
    label="Sun",
)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.legend(
    bbox_to_anchor=(1, 0.7), frameon=False, loc=0, fontsize=15, markerscale=2
)

plt.show()

Side view of the Galactic plane.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulated all",
)

ax.plot(
    x[intercepted_radio],
    z[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    x[idx_PMPS],
    z[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    x[idx_SMPS],
    z[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    x[idx_HTRU_low_mid],
    z[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.plot(
    x[idx_HTRU_high],
    z[idx_HTRU_high],
    linestyle="None",
    marker="o",
    color="tab:olive",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU high",
)

ax.plot(
    0.0,
    0.02,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=6,
    label="Sun",
)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.legend(
    bbox_to_anchor=(1, 0.7), frameon=False, loc=0, fontsize=15, markerscale=2
)

plt.show()

Histrograming the pulsars' radial positions and comparing to the underlying initial position PDF.

In [ ]:
def pdf_r(r: float) -> float:
    """
    The Milky Way's stellar radial density in the galactic plane according
    to eq. (15) of Yusifov & Küçük (2004).

    Args:
        r (float): Distance from the galactic center in [kpc].

    Returns:
        float: Stellar radial density in [1/kpc].
    """

    # Here we keep R_sun = 8.5 kpc for consistency with the results
    # of Yusifov & Küçük (2004)
    rsun = 8.5  # Sun's distance from the galactic center in [kpc].
    A = 37.6  # +- 1.90 [1/kpc^2]
    a = 1.64  # +-0.11
    b = 4.01  # +-0.24
    r1 = 0.55  # +- 0.10 [kpc]

    # Stellar surface density following eq. (15) of Yusifov & Küçük (2004).
    rho = (
        A
        * ((r + r1) / (rsun + r1)) ** a
        * np.exp(-b * (r - rsun) / (rsun + r1))
    )

    # Multiply the stellar surface density with the area element in polar coordinates.
    pdf_r = 2 * np.pi * r * rho

    return pdf_r

For normalization purposes, determine the area underneath the theoretical PDF curve.

In [ ]:
pdf_area = quad(pdf_r, 0, 100)[0]
print(pdf_area)

In [ ]:
r = np.sqrt(x**2 + y**2)
r_bins = np.linspace(0.0, 30.0, 31)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    r,
    bins=r_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Simulated all",
)
ax.hist(
    r[intercepted_radio],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    alpha=1,
    label="Intercepting our LOS",
)
ax.hist(
    r[idx_PMPS],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="Detected by PMPS",
)
ax.hist(
    r[idx_SMPS],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Detected by SMPS",
)
ax.hist(
    r[idx_HTRU_low_mid],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label="Detected by HTRU low and mid",
)
ax.hist(
    r[idx_HTRU_high],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    alpha=1,
    label="Detected by HTRU high",
)

ax.plot(
    r_bins,
    pdf_r(r_bins) / pdf_area * 1.0e5,
    linestyle="--",
    lw=4,
    color="black",
    alpha=1,
    label="Initial YK04",
)
plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"Number of NSs")
plt.xlim(0.0, 30.0)
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Histrograming the pulsars' $z$ position and comparing to underlying PDF.

In [ ]:
def pdf_z(z: float) -> float:
    """
    Probability density function for the height from the galactic equatorial plane
    according to eq. (2) in Gullon et al. (2014).

    Args:
        z (float): Distance from the galactic plane in [kpc].

    Returns:
        float: Distribution of stars per kpc in z direction.
    """

    # We use an exponential distribution as given by Wainscoat et al. (1992)
    # and choose a mean scale height characteristic for a young distribution as
    # obtained by Gullon et al. (2014).

    h_c = 0.18
    pdf_z = 1.0 / h_c * np.exp(-z / h_c)

    return pdf_z

In [ ]:
pdf_area = quad(pdf_z, 0, 5)[0]
print(pdf_area)

In [ ]:
z_bins = np.linspace(0.0, 5.0, 31)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    z,
    bins=z_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Simulated all",
)
ax.hist(
    z[intercepted_radio],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    alpha=1,
    label="Intercepting our LOS",
)
ax.hist(
    z[idx_PMPS],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="Detected by PMPS",
)
ax.hist(
    z[idx_SMPS],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Detected by SMPS",
)
ax.hist(
    z[idx_HTRU_low_mid],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label="Detected by HTRU low and mid",
)
ax.hist(
    z[idx_HTRU_high],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    alpha=1,
    label="Detected by HTRU high",
)

ax.plot(
    z_bins,
    pdf_z(z_bins) * 1.0e4,
    linestyle="--",
    lw=4,
    color="black",
    alpha=1,
    label="Theoretical initial PDF",
)
plt.xlabel(r"$z$ [kpc]")
plt.ylabel(r"Number of NS")
plt.xlim(0.0, 5.0)
plt.ylim(0.1, 1.0e6)
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Histograming the pulsar heliocentric distances.

In [ ]:
dist_bins = np.linspace(0.0, 30.0, 31)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    dist,
    bins=dist_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label="Simulated all",
)
ax.hist(
    dist[intercepted_radio],
    bins=dist_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    alpha=1,
    label="Intercepting our LOS",
)
ax.hist(
    dist[idx_PMPS],
    bins=dist_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="Detected by PMPS",
)
ax.hist(
    dist[idx_SMPS],
    bins=dist_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Detected by SMPS",
)
ax.hist(
    dist[idx_HTRU_low_mid],
    bins=dist_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label="Detected by HTRU low and mid",
)
ax.hist(
    dist[idx_HTRU_high],
    bins=dist_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    alpha=1,
    label="Detected by HTRU high",
)

plt.xlabel(r"$d_{\odot}$ [kpc]")
plt.ylabel(r"Number of NS")
# plt.xlim(0.0, 5.0)
plt.ylim(0.1, 1.0e6)
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

## Proper velocity comparison

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
x_bins = np.linspace(-1500.0, 1500.0, 51)

ax.hist(
    v_r,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"$v_r$",
)
ax.hist(
    v_phi,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"$v_{\phi}$",
)
ax.hist(
    v_z,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"$v_{z}$",
)
ax.set_xlabel(r"Velocity components [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0, fontsize=20)

plt.show()

Distribution of the total velocity magnitude.

In [ ]:
def pdf_kick_velocity_maxwell(v: float) -> float:
    """
    Maxwell probability density function for the neutron stars' initial kick
    velocity magnitude following Hobbs et al. (2005).

    Args:
        v (float): Initial kick velocity magnitude in [km/s].

    Returns:
        float: Stellar kick velocity distribution in [1/(km/s)].
    """
    sigma = 265.0
    pdf_vk = (
        np.sqrt(2 / np.pi)
        * v**2
        / (sigma**3)
        * np.exp(-(v**2) / (2 * sigma**2))
    )

    return pdf_vk

In [ ]:
v_tot = np.sqrt(v_r**2 + v_phi**2 + v_z**2)

fig, ax = plt.subplots(figsize=(15, 8))
v_bins = np.linspace(0, 1500.0, 31)

ax.hist(
    v_tot,
    bins=v_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    alpha=1,
    label=r"Simulated all",
)
ax.hist(
    v_tot[intercepted_radio],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    alpha=1,
    label=r"Intercepting our LOS",
)
ax.hist(
    v_tot[idx_PMPS],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Detected by PMPS",
)
ax.hist(
    v_tot[idx_SMPS],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Detected by SMPS",
)
ax.hist(
    v_tot[idx_HTRU_low_mid],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"Detected by HTRU low and mid",
)
ax.hist(
    v_tot[idx_HTRU_high],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    alpha=1,
    label=r"Detected by HTRU high",
)

ax.plot(
    v_bins,
    pdf_kick_velocity_maxwell(v_bins) * 5e6,
    linestyle="--",
    lw=4,
    color="black",
    alpha=1,
    label=r"Maxwell $\sigma = 265$ km s$^{-1}$",
)
ax.set_xlabel(r"Total velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.yscale("log")
plt.ylim(0.1, 1.0e5)
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

## Magneto-rotational information

Histograming the periods and magnetic fields.

In [ ]:
def Gaussian(x, mean, sigma):
    y = (
        1
        / (sigma * np.sqrt(2 * np.pi))
        * np.exp(-((x - mean) ** 2) / (2 * sigma**2))
    )
    return y

In [ ]:
P_bins = np.logspace(-2.0, 2.5, 31)
P_grid = np.logspace(-2.0, 2.5, 1000)
print(max(P), min(P))

In [ ]:
cfg["P_initial_mean"] = 0.3
cfg["P_initial_sigma"] = 0.2

In [ ]:
pdf_P_initial_area = quad(
    Gaussian, 0, 100, args=(cfg["P_initial_mean"], cfg["P_initial_sigma"])
)[0]
print(pdf_P_initial_area)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    P,
    bins=P_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulated all",
)
ax.hist(
    P[intercepted_radio],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    label="Intercepting our LOS",
)
ax.hist(
    P[idx_PMPS],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Detected by PMPS",
)
ax.hist(
    P[idx_SMPS],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Detected by SMPS",
)
ax.hist(
    P[idx_HTRU_low_mid],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Detected by HTRU low and mid",
)
ax.hist(
    P[idx_HTRU_high],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    label="Detected by HTRU high",
)

ax.plot(
    P_grid,
    Gaussian(P_grid, cfg["P_initial_mean"], cfg["P_initial_sigma"])
    / pdf_P_initial_area
    * 5.0e4,
    linestyle="--",
    lw=4,
    color="black",
    label="Theoretical initial Normal",
)
ax.plot(
    P_grid,
    Gaussian(
        np.log10(P_grid),
        cfg["P_initial_log10_mean"],
        cfg["P_initial_log10_sigma"],
    )
    * 5.0e4,
    linestyle=":",
    lw=4,
    color="black",
    label="Theoretical initial Log-normal",
)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
# plt.xlim(0., 50.0)
plt.ylim(0.1, 2.0e5)
plt.xscale("log")
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
B_edges = np.logspace(7.0, 17.0, 51)
print(max(np.log10(B)), min(np.log10(B)))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    B,
    bins=B_edges,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulation all",
)
ax.hist(
    B[intercepted_radio],
    bins=B_edges,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    label="Intercepting our LOS",
)
ax.hist(
    B[idx_PMPS],
    bins=B_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Detected by PMPS",
)
ax.hist(
    B[idx_SMPS],
    bins=B_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Detected by SMPS",
)
ax.hist(
    B[idx_HTRU_low_mid],
    bins=B_edges,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Detected by HTRU low and mid",
)
ax.hist(
    B[idx_HTRU_high],
    bins=B_edges,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    label="Detected by HTRU high",
)

ax.plot(
    B_edges,
    Gaussian(
        np.log10(B_edges),
        cfg["B_initial_log10_mean"],
        cfg["B_initial_log10_sigma"],
    )
    * 9.0e3,
    linestyle="--",
    lw=4,
    color="black",
    label="Theoretical initial",
)

plt.xlabel(r"$B$ [G]")
plt.ylabel(r"Number of NSs")
plt.ylim(0.1, 5.0e4)
plt.xscale("log")
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    P,
    B,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulation all",
)

ax.plot(
    P[intercepted_radio],
    B[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    P[idx_PMPS],
    B[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    P[idx_SMPS],
    B[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    P[idx_HTRU_low_mid],
    B[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.plot(
    P[idx_HTRU_high],
    B[idx_HTRU_high],
    linestyle="None",
    marker="o",
    color="tab:olive",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU high",
)

ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"$B$ [G]")
plt.xscale("log")
plt.yscale("log")
plt.legend(
    bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20, markerscale=3
)

plt.show()

## Beaming characteristics

Here, we study the effect of beaming in the radio emission and how the evolution in the inclination angle affects this beamed emission. We compare the beaming fraction (fraction of pulsars intercepting our line of sight) in our simulation with the expected beaming fraction from the empirical formula (eq. 15) in [Tauris and Manchester (1998)](https://ui.adsabs.harvard.edu/abs/1998MNRAS.298..625T/abstract).
This empirical formula is derived from observational data and depends only on the spin period of a neutron star. In our simulation instead the beam fraction depends also on other variables like the inclination angle. Indeed neutron stars with small inclination angle have a smaller chance to intercept our line of sight with their radio beam compared to stars with larger inclination angle, regardless of the spin period value.

In [ ]:
def beam_fraction_theoretical(P: np.ndarray) -> np.ndarray:
    """
    Fraction of pulsar beaming towards us as a function of their spin period.
    Empirical fit from Tauris and Manchester (1998).

    Args:
        P (np.ndarray): Array of spin periods of the pulsars in [s].

    Returns:
        (np.ndarray): Beaming fraction.
    """

    f_b = 0.09 * (np.log10(P) - 1) ** 2 + 0.03

    return f_b

In [ ]:
def los_intercept(f_b: np.ndarray) -> np.ndarray:
    """
    Evaluate if the radio beam intercepts the line of sight (LOS),
    assuming random orientation of the LOS with respect to the rotation axis.

    Args:
        f_p (np.ndarray): Array of beam fraction of the pulsars.

    Returns:
        (np.ndarray): Array of boolean variables: true if the radio beam
            intercept the LOS and false if not.
    """
    rand = np.random.uniform(0.0, 1.0, len(f_b))
    detected = np.zeros(len(f_b), dtype=bool)
    for i in range(len(f_b)):
        if rand[i] < f_b[i]:
            detected[i] = True

    return detected

In [ ]:
# Compute the probability for a pulsar to point towards us
# as a function of its spin period from empirical formula in Tauris and Manchester (1998).
f_b = beam_fraction_theoretical(P)
intercepted_th = los_intercept(f_b)

P_bin_edges = np.logspace(np.log10(0.03), np.log10(20), 11)
P_bin_centers = 0.5 * (P_bin_edges[1:] + P_bin_edges[:-1])

# For a given period, compute the fraction of pulsars pointing towards us
# from the simulation and from the empirical formula above.
(
    h1,
    _,
) = np.histogram(P, bins=P_bin_edges)
(
    h2,
    _,
) = np.histogram(P[intercepted_radio], bins=P_bin_edges)
(
    h3,
    _,
) = np.histogram(P[intercepted_th], bins=P_bin_edges)

bf_simulation = h2 / h1
bf_theoretical = h3 / h1
bf_curve = beam_fraction_theoretical(P_bin_centers)

In [ ]:
# Plot the beaming fraction curve as a function of period.
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    P_bin_centers,
    bf_simulation,
    linestyle="-",
    linewidth=3,
    color="tab:cyan",
    label="Simulation",
)
ax.plot(
    P_bin_centers,
    bf_theoretical,
    linestyle=":",
    linewidth=3,
    color="tab:cyan",
    label="Derived from T & M (1998)",
)
ax.plot(
    P_bin_centers,
    bf_curve,
    linestyle="--",
    linewidth=3,
    color="black",
    label="Model T & M (1998)",
)

plt.xscale("log")
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Beaming fraction")
plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
chi_bins = np.linspace(0, np.pi / 2, 31)
print(max(chi), min(chi))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    chi,
    bins=chi_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulated all",
)
ax.hist(
    chi[intercepted_radio],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    label="Intercepting our LOS",
)
ax.hist(
    chi[intercepted_th],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    ls=":",
    label="Intercepting our LOS (T&M 1998)",
)
ax.hist(
    chi[idx_PMPS],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Detected by PMPS",
)
ax.hist(
    chi[idx_SMPS],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Detected by SMPS",
)
ax.hist(
    chi[idx_HTRU_low_mid],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Detected by HTRU low and mid",
)
ax.hist(
    chi[idx_HTRU_high],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    label="Detected by HTRU high",
)

ax.plot(
    chi_bins,
    np.sin(chi_bins) * 5.0e3,
    linestyle="--",
    lw=4,
    color="black",
    label="Theoretical initial",
)
plt.xlabel(r"$\chi$ [rad]")
plt.ylabel(r"Normalized PDF")
plt.xlim(0.0, np.pi / 2)
plt.ylim(0.1, 1.0e5)
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Note that by using the empirical law from Tauris \& Manchester (1998), we erroneously select more stars that have radio beams that are aligned with the rotation axis and pointing at us. Instead, these should be less likely to be pointing at us. This is due to the fact that the empirical formula from Tauris \& Manchester (1998) only depends on the spin period, but in reality it should depend also on the inclination angle.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    P,
    chi,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulation all",
)

ax.plot(
    P[intercepted_radio],
    chi[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    P[idx_PMPS],
    chi[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    P[idx_SMPS],
    chi[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    P[idx_HTRU_low_mid],
    chi[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.plot(
    P[idx_HTRU_high],
    chi[idx_HTRU_high],
    linestyle="None",
    marker="o",
    color="tab:olive",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU high",
)

ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"$\chi$ [rad]")
plt.xscale("log")
plt.legend(
    bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20, markerscale=3
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    B,
    chi,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulation all",
)

ax.plot(
    B[intercepted_radio],
    chi[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    B[idx_PMPS],
    chi[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    B[idx_SMPS],
    chi[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    B[idx_HTRU_low_mid],
    chi[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.plot(
    B[idx_HTRU_high],
    chi[idx_HTRU_high],
    linestyle="None",
    marker="o",
    color="tab:olive",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU high",
)

ax.set_xlabel(r"$B$ [G]")
ax.set_ylabel(r"$\chi$ [rad]")
ax.set_xscale("log")
plt.legend(
    bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20, markerscale=3
)

plt.show()

## $P-\dot{P}$ diagram

Plotting the $P-\dot{P}$ diagram of the final pulsar population, representing a snapshot at the current time.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    P,
    P_dot,
    linestyle="None",
    marker="o",
    color="darkgray",
    markersize=1,
    alpha=0.1,
    rasterized=True,
    label="Simulation all",
)
ax.plot(
    P[intercepted_radio],
    P_dot[intercepted_radio],
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=2,
    alpha=0.1,
    rasterized=True,
    label="Intercepting our LOS",
)
ax.plot(
    P[idx_PMPS],
    P_dot[idx_PMPS],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by PMPS",
)
ax.plot(
    P[idx_SMPS],
    P_dot[idx_SMPS],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by SMPS",
)
ax.plot(
    P[idx_HTRU_low_mid],
    P_dot[idx_HTRU_low_mid],
    linestyle="None",
    marker="o",
    color="tab:green",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.plot(
    P[idx_HTRU_high],
    P_dot[idx_HTRU_high],
    linestyle="None",
    marker="o",
    color="tab:olive",
    markersize=2,
    alpha=1.0,
    rasterized=True,
    label="Detected by HTRU high",
)

ax.set_xscale("log")
ax.set_yscale("log")
# ax.set_ylim(1.e-22, 1.e-20)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
plt.legend(
    bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20, markerscale=3
)

plt.show()

# Radio emission

In [ ]:
# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * cfg["NS_mass"] * cfg["NS_radius"] ** 2

# Compute spin down power.
Erot_dot = NS_inertia * (2.0 * np.pi) ** 2 * P_dot / (P**3)

# Compute pseudo luminosity in [mJy kpc^2].
pseudo_L_radio_PMPS = S_radio_PMPS * dist[idx_PMPS] ** 2 * 1000
pseudo_L_radio_SMPS = S_radio_SMPS * dist[idx_SMPS] ** 2 * 1000
pseudo_L_radio_HTRU_low_mid = (
    S_radio_HTRU_low_mid * dist[idx_HTRU_low_mid] ** 2 * 1000
)
pseudo_L_radio_HTRU_high = S_radio_HTRU_high * dist[idx_HTRU_high] ** 2 * 1000

pseudo_L_radio = np.concatenate((pseudo_L_radio_PMPS, pseudo_L_radio_SMPS))

Histograming the spin-down power.

In [ ]:
Erot_dot_bins = np.logspace(20, 40, 51)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    Erot_dot,
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulated all",
)
ax.hist(
    Erot_dot[intercepted_radio],
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    label="Intercepting our LOS",
)
ax.hist(
    Erot_dot[idx_PMPS],
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Detected by PMPS",
)
ax.hist(
    Erot_dot[idx_SMPS],
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Detected by SMPS",
)
ax.hist(
    Erot_dot[idx_HTRU_low_mid],
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Detected by HTRU low and mid",
)
ax.hist(
    Erot_dot[idx_HTRU_high],
    bins=Erot_dot_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    label="Detected by HTRU high",
)

plt.xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.set_yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Histograming the radio luminosities.

In [ ]:
L_radio_bins = np.logspace(20, 40, 51)
print(L_radio_bol)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    L_radio_bol,
    bins=L_radio_bins,
    histtype="step",
    edgecolor="darkgray",
    lw=4,
    label="Simulated all",
)
ax.hist(
    L_radio_bol[intercepted_radio],
    bins=L_radio_bins,
    histtype="step",
    edgecolor="tab:cyan",
    lw=4,
    label="Intercepting our LOS",
)
ax.hist(
    L_radio_bol[idx_PMPS],
    bins=L_radio_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Detected by PMPS",
)
ax.hist(
    L_radio_bol[idx_SMPS],
    bins=L_radio_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Detected by SMPS",
)
ax.hist(
    L_radio_bol[idx_HTRU_low_mid],
    bins=L_radio_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Detected by HTRU low and mid",
)
ax.hist(
    L_radio_bol[idx_HTRU_high],
    bins=L_radio_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    label="Detected by HTRU high",
)
plt.xlabel(r"$L_{\rm radio}$ [erg s$^{-1}$ Hz$^{-1}$]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.set_yscale("log")
plt.legend(frameon=False, loc=0, fontsize=20)

plt.show()

Plot histogram of observed radio fluxes.

In [ ]:
S_radio_bins = np.logspace(-8, 2, 51)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S_radio_PMPS,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Detected by PMPS",
)
ax.hist(
    S_radio_SMPS,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Detected by SMPS",
)
ax.hist(
    S_radio_HTRU_low_mid,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    label="Detected by HTRU low and mid",
)
ax.hist(
    S_radio_HTRU_high,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:olive",
    lw=4,
    label="Detected by HTRU high",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.set_yscale("log")
plt.legend(frameon=False, loc=0, fontsize=20)

plt.show()

Compare the relationship between the radio luminosity/pseudo luminosity and the spin-down power.

In [ ]:
def linear_relation(x: np.ndarray, a: float, b: float) -> np.ndarray:
    """
    A first degree linear equation.

    Args:
        x (np.ndarray): Input array of values.
        a (float): Slope of the linear relation.
        b (float): Intercept of the linear relation.

    Returns:
        (np.ndarray): Output values of the linear relation.
    """

    y = a * x + b

    return y

In [ ]:
logErot_dot_ps = np.log10(Erot_dot[idx_radio_det])
logL_radio_ps = np.log10(pseudo_L_radio)

# Compute the barycenter of the data.
xm_ps = np.mean(logErot_dot_ps)
ym_ps = np.mean(logL_radio_ps)

# Fit the data with curve_fit: popt[0] and popt[1] are the optimal a and b parameters;
# pcov is the covariance matrix.
popt_ps, pcov_ps = curve_fit(
    linear_relation,
    logErot_dot_ps - xm_ps,
    logL_radio_ps - ym_ps,
    [0.1, 1.0e-10],
)

# Evaluate the 1-sigma error on the parameters
perr_ps = np.sqrt(np.diag(pcov_ps))

a_ps = popt_ps[0]
b_ps = popt_ps[1]
a_err_ps = perr_ps[0]
b_err_ps = perr_ps[1]

x_grid_ps = np.linspace(27.0, 40.0, 1000)
best_fit_ps = 10 ** (a_ps * (x_grid_ps - xm_ps) + b_ps + ym_ps)

print(
    f"Results of the linear fit for the pseudo luminosity in log: \n a = {a_ps} +- {a_err_ps}; \n b = {b_ps} +- {b_err_ps}"
)

In [ ]:
logErot_dot_true = np.log10(Erot_dot[idx_radio_det])
logL_radio_true = np.log10(L_radio_bol[idx_radio_det])

# Compute the barycenter of the data.
xm_true = np.mean(logErot_dot_true)
ym_true = np.mean(logL_radio_true)

# Fit the data with curve_fit: popt[0] and popt[1] are the optimal a and b parameters;
# pcov is the covariance matrix.
popt_true, pcov_true = curve_fit(
    linear_relation,
    logErot_dot_true - xm_true,
    logL_radio_true - ym_true,
    [0.1, 1.0e-10],
)

# Evaluate the 1-sigma error on the parameters
perr_true = np.sqrt(np.diag(pcov_true))

a_true = popt_true[0]
b_true = popt_true[1]
a_err_true = perr_true[0]
b_err_true = perr_true[1]

x_grid_true = np.linspace(27.0, 40.0, 1000)
best_fit_true = 10 ** (a_true * (x_grid_true - xm_true) + b_true + ym_true)

print(
    f"Results of the linear fit for the true luminosity in log: \n a = {a_true} +- {a_err_true}; \n b = {b_true} +- {b_err_true}"
)

In [ ]:
fig, ax1 = plt.subplots(figsize=(18, 12))
ax1.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax1.set_ylabel(r"$S d^2$ [mJy kpc$^2$]")

ax2 = ax1.twinx()
ax2.set_ylabel(r"$L_{\rm radio}$ [erg s$^{-1}$ Hz$^{-1}$]")
ax2.loglog(
    Erot_dot[idx_radio_det],
    L_radio_bol[idx_radio_det],
    "o",
    color="black",
    ms=6,
    alpha=0.5,
    rasterized=True,
    label="True luminosity",
)
ax2.plot(
    10**x_grid_true,
    best_fit_true,
    linestyle="-",
    linewidth=3,
    color="black",
    label="Fit true luminosity",
)

ax1.loglog(
    Erot_dot[idx_PMPS],
    pseudo_L_radio_PMPS,
    "x",
    color="tab:red",
    ms=10,
    alpha=1,
    rasterized=True,
    label="Pseudo luminosity PMPS",
)
ax1.loglog(
    Erot_dot[idx_SMPS],
    pseudo_L_radio_SMPS,
    "x",
    color="tab:blue",
    ms=10,
    alpha=1,
    rasterized=True,
    label="Pseudo luminosity SMPS",
)
ax1.loglog(
    Erot_dot[idx_HTRU_low_mid],
    pseudo_L_radio_HTRU_low_mid,
    "x",
    color="tab:green",
    ms=10,
    alpha=1,
    rasterized=True,
    label="Pseudo luminosity HTRU low and mid",
)
ax1.loglog(
    Erot_dot[idx_HTRU_high],
    pseudo_L_radio_HTRU_high,
    "x",
    color="tab:olive",
    ms=10,
    alpha=1,
    rasterized=True,
    label="Pseudo luminosity HTRU high",
)
ax1.plot(
    10**x_grid_ps,
    best_fit_ps,
    linestyle="-",
    linewidth=3,
    color="darkgray",
    label="Fit pseudo-luminosity",
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
plt.legend(
    lines1 + lines2,
    labels1 + labels2,
    frameon=False,
    loc=2,
    fontsize=20,
    markerscale=3,
)
plt.show(block=False)

Plot radio efficiencies as a function of the spin-down power.

In [ ]:
# Consider the luminosity integrated on a frequency range of 1.374 GHz,
# assuming it is constant in that frequency range.
delta_nu = 1.374e9
eff_radio = L_radio_bol / Erot_dot
eff_pseudo_radio_PMPS = (
    pseudo_L_radio_PMPS
    * const.MILLIJY_TO_ERG
    * const.KPC_TO_CM**2
    * delta_nu
    / Erot_dot[idx_PMPS]
)
eff_pseudo_radio_SMPS = (
    pseudo_L_radio_SMPS
    * const.MILLIJY_TO_ERG
    * const.KPC_TO_CM**2
    * delta_nu
    / Erot_dot[idx_SMPS]
)
eff_pseudo_radio_HTRU_low_mid = (
    pseudo_L_radio_HTRU_low_mid
    * const.MILLIJY_TO_ERG
    * const.KPC_TO_CM**2
    * delta_nu
    / Erot_dot[idx_HTRU_low_mid]
)
eff_pseudo_radio_HTRU_high = (
    pseudo_L_radio_HTRU_high
    * const.MILLIJY_TO_ERG
    * const.KPC_TO_CM**2
    * delta_nu
    / Erot_dot[idx_HTRU_high]
)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))
ax.set_xlabel(r"$\dot{E}_{\rm rot}$ [erg s$^{-1}$]")
ax.set_ylabel(r"Efficiency")
ax.loglog(
    Erot_dot[idx_radio_det],
    eff_radio[idx_radio_det],
    "o",
    color="black",
    ms=6,
    alpha=0.5,
    rasterized=True,
    label="True luminosity",
)
ax.loglog(
    Erot_dot[idx_PMPS],
    eff_pseudo_radio_PMPS,
    "x",
    color="tab:red",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Pseudo luminosity PMPS",
)
ax.loglog(
    Erot_dot[idx_SMPS],
    eff_pseudo_radio_SMPS,
    "x",
    color="tab:blue",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Pseudo luminosity SMPS",
)
ax.loglog(
    Erot_dot[idx_HTRU_low_mid],
    eff_pseudo_radio_HTRU_low_mid,
    "x",
    color="tab:green",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Pseudo luminosity HTRU low and mid",
)
ax.loglog(
    Erot_dot[idx_HTRU_high],
    eff_pseudo_radio_HTRU_high,
    "x",
    color="tab:olive",
    ms=6,
    alpha=1,
    rasterized=True,
    label="Pseudo luminosity HTRU high",
)
ax.axhline(y=1, color="black", linestyle="-", linewidth=2)
ax.axhline(y=0.1, color="black", linestyle="--", linewidth=2)
plt.legend(frameon=False, loc=3, fontsize=20, markerscale=3)

plt.show(block=False)

Radio pulse width.

In [ ]:
w_int_deg = w_int / P * 360.0
w_obs_deg_PMPS = w_PMPS / P[idx_PMPS] * 360.0
w_obs_deg_SMPS = w_SMPS / P[idx_SMPS] * 360.0
w_obs_deg_HTRU_low_mid = w_HTRU_low_mid / P[idx_HTRU_low_mid] * 360.0
w_obs_deg_HTRU_high = w_HTRU_high / P[idx_HTRU_high] * 360.0
w_obs_deg = np.concatenate((w_obs_deg_PMPS, w_obs_deg_SMPS))

In [ ]:
def fit_MG2011(P: np.ndarray) -> np.ndarray:
    """
    Fit of the W50 pulse widths of double pulse pulsars as a function of their spin period
    from Maciesiak and Gil (2011).

    Args:
        P (np.ndarray): Spin period in [s].

    Returns:
        (np.ndarray): Value of the W50 predicted by the model in [deg].
    """
    w50 = 2.5 * P ** (-0.5)

    return w50


# Radio beam aperture model used in the simulation.
w_theory = 2 * er.beam_aperture(P_grid) / np.pi * 180.0

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

ax.loglog(
    P[idx_PMPS],
    # w_int_deg[idx_PMPS],
    w_obs_deg_PMPS,
    ".",
    color="tab:red",
    ms=6,
    rasterized=True,
    label="Detected by PMPS",
)
ax.loglog(
    P[idx_SMPS],
    # w_int_deg[idx_SMPS],
    w_obs_deg_SMPS,
    ".",
    color="tab:blue",
    ms=6,
    rasterized=True,
    label="Detected by SMPS",
)
ax.loglog(
    P[idx_HTRU_low_mid],
    # w_int_deg[idx_HTRU_low_mid],
    w_obs_deg_HTRU_low_mid,
    ".",
    color="tab:purple",
    ms=6,
    rasterized=True,
    label="Detected by HTRU low and mid",
)
ax.loglog(
    P[idx_HTRU_high],
    # w_int_deg[idx_SMPS],
    w_obs_deg_HTRU_high,
    ".",
    color="tab:olive",
    ms=6,
    rasterized=True,
    label="Detected by HTRU high",
)

ax.loglog(
    P_grid,
    fit_MG2011(P_grid),
    "--",
    color="black",
    lw=3,
    label=r"$w_{50} = 2.5^{\circ} P^{-0.5}$",
)
ax.loglog(
    P_grid, w_theory, "-", color="black", lw=3, label=r"Theoretical model"
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$w$ [deg]")

plt.legend(frameon=False, loc=3, fontsize=20, markerscale=3)
plt.show(block=False)